# 03. Pandas 집계 분석 - 실습 문제

## 실습 안내
- 총 10개 문제
- 설비별, 제품별, 시간대별 집계 분석
- groupby, pivot_table, crosstab 활용
- 실제 제조 현장의 분석 시나리오

## 데이터 로드 및 전처리

In [2]:
import pandas as pd
import numpy as np

# 데이터 불러오기
production_df = pd.read_csv('../data/05_production.csv', encoding='utf-8-sig')
quality_df = pd.read_csv('../data/07_quality_inspection.csv', encoding='utf-8-sig', na_values=['\\N'])
equipment_df = pd.read_csv('../data/01_equipment.csv', encoding='utf-8-sig')
operation_df = pd.read_csv('../data/06_equipment_operation.csv', encoding='utf-8-sig')

# 기본 전처리
production_df['production_date'] = pd.to_datetime(production_df['production_date'])
production_df['defect_rate'] = (production_df['defect_quantity'] / production_df['actual_quantity'] * 100).round(2)
quality_df['inspection_time'] = pd.to_datetime(quality_df['inspection_time'])

print("데이터 로드 완료!")
print(f"생산: {len(production_df):,}건")
print(f"품질: {len(quality_df):,}건")
print(f"설비: {len(equipment_df):,}건")
print(f"설비운영: {len(operation_df):,}건")

데이터 로드 완료!
생산: 1,872건
품질: 37,417건
설비: 5건
설비운영: 3,304건


---
## 문제 1: 설비별 생산 통계

**시나리오**: 각 설비의 생산 성과를 종합적으로 분석하세요.

**요구사항**:
1. 설비별로 다음 지표 계산:
   - 생산 건수 (count)
   - 총 생산량 (actual_quantity의 sum)
   - 평균 생산량 (actual_quantity의 mean)
   - 총 불량수 (defect_quantity의 sum)
   - 평균 불량률 (defect_rate의 mean)
2. 총 생산량 기준 내림차순 정렬
3. 소수점 2자리로 반올림



In [3]:
# 1번 문제 확인


In [4]:
production_df.columns

Index(['production_id', 'equipment_id', 'product_code', 'production_date',
       'start_time', 'end_time', 'target_quantity', 'actual_quantity',
       'good_quantity', 'defect_quantity', 'cycle_time', 'work_order_no',
       'lot_no', 'operator_id', 'shift', 'created_at', 'updated_at',
       'defect_rate'],
      dtype='object')

In [5]:
production_df.groupby('equipment_id').agg({'actual_quantity':'count'})

,actual_quantity
equipment_id,
ASM-001,234
INJ-001,262
INJ-002,430
PRESS-001,468
PRESS-002,478


In [6]:
production_df.groupby('equipment_id').agg({'actual_quantity':'sum'})

,actual_quantity
equipment_id,
ASM-001,22485
INJ-001,28163
INJ-002,51958
PRESS-001,52069
PRESS-002,51929


In [7]:
production_df.groupby('equipment_id').agg({'actual_quantity':'mean'})

,actual_quantity
equipment_id,
ASM-001,96.089744
INJ-001,107.492366
INJ-002,120.832558
PRESS-001,111.258547
PRESS-002,108.638075


In [8]:
production_df.groupby('equipment_id').agg({'defect_rate':'mean'})

,defect_rate
equipment_id,
ASM-001,12.116453
INJ-001,10.759046
INJ-002,8.701302
PRESS-001,9.910855
PRESS-002,10.675565


In [9]:
production_df.groupby('equipment_id').size().sort_values(ascending=False)

equipment_id
PRESS-002    478
PRESS-001    468
INJ-002      430
INJ-001      262
ASM-001      234
dtype: int64

In [10]:
(production_df.groupby('equipment_id').size().sort_values(ascending=False)).round(2)

equipment_id
PRESS-002    478
PRESS-001    468
INJ-002      430
INJ-001      262
ASM-001      234
dtype: int64

---
## 문제 2: 제품별 품질 분석

**시나리오**: 제품별 품질 수준을 파악하여 문제가 있는 제품을 찾으세요.

**요구사항**:
1. 제품별로 다음 지표 계산:
   - 생산 건수
   - 총 생산량
   - 평균 불량률
   - 최대 불량률
   - 최소 불량률
2. 평균 불량률이 높은 순서로 정렬



In [11]:
# 여기에 코드 작성
production_df.columns

Index(['production_id', 'equipment_id', 'product_code', 'production_date',
       'start_time', 'end_time', 'target_quantity', 'actual_quantity',
       'good_quantity', 'defect_quantity', 'cycle_time', 'work_order_no',
       'lot_no', 'operator_id', 'shift', 'created_at', 'updated_at',
       'defect_rate'],
      dtype='object')

In [12]:
production_df.groupby('product_code').size()

product_code
BUMPER-A    648
DASH-C      583
DOOR-B      641
dtype: int64

In [13]:
production_df.groupby('product_code').agg({'actual_quantity':'sum'})

,actual_quantity
product_code,
BUMPER-A,71135
DASH-C,64741
DOOR-B,70728


In [14]:
production_df.groupby('product_code').agg({'defect_rate':['mean','min','max']})

defect_rate            
                    mean  min    max
product_code                        
BUMPER-A       10.376590  2.0  19.48
DASH-C         10.121681  2.0  19.35
DOOR-B         10.158986  2.0  20.00

In [15]:
production_df.groupby('product_code') \
             .agg({'defect_rate': ['mean', 'min', 'max']}) \
             .sort_values(by=('defect_rate', 'mean'), ascending=False)


defect_rate            
                    mean  min    max
product_code                        
BUMPER-A       10.376590  2.0  19.48
DOOR-B         10.158986  2.0  20.00
DASH-C         10.121681  2.0  19.35

---
## 문제 3: 교대조별 생산 효율 비교

**시나리오**: 주간조와 야간조의 생산 효율을 비교 분석하세요.

**요구사항**:
1. 교대조(shift)별로 다음 지표 계산:
   - 생산 건수
   - 평균 생산량
   - 평균 불량률
   - 평균 사이클 타임
   - 목표 달성률 평균 (actual_quantity / target_quantity * 100)
2. 결과 해석: 어느 교대조가 더 효율적인가?



In [16]:
# 여기에 코드 작성
production_df.columns

Index(['production_id', 'equipment_id', 'product_code', 'production_date',
       'start_time', 'end_time', 'target_quantity', 'actual_quantity',
       'good_quantity', 'defect_quantity', 'cycle_time', 'work_order_no',
       'lot_no', 'operator_id', 'shift', 'created_at', 'updated_at',
       'defect_rate'],
      dtype='object')

In [17]:
production_df.groupby('shift').size()

shift
DAY      936
NIGHT    936
dtype: int64

In [18]:
production_df.groupby('shift').agg({'actual_quantity':'mean'})

,actual_quantity
shift,
DAY,109.711538
NIGHT,111.019231


In [19]:
production_df.groupby('shift').agg({'defect_rate':'mean'})

,defect_rate
shift,
DAY,8.815769
NIGHT,11.629615


In [20]:
production_df.groupby('shift').agg({'cycle_time':'mean'})

,cycle_time
shift,
DAY,76.63562
NIGHT,76.63563


In [21]:
production_df['목표 달성률 평균']=production_df['actual_quantity']/production_df['target_quantity']*100

In [22]:
production_df.groupby('shift').agg({'목표 달성률 평균':'mean'})

,목표 달성률 평균
shift,
DAY,94.818320
NIGHT,96.640882


---
## 문제 4: 설비 + 제품 복합 분석

**시나리오**: 각 설비가 어떤 제품을 얼마나 생산하는지 분석하세요.

**요구사항**:
1. 설비 + 제품별로 다음 집계:
   - 생산 건수
   - 총 생산량
   - 평균 불량률
2. 멀티인덱스 결과에서 특정 설비(예: INJ-001)의 제품별 데이터만 추출



In [23]:
production_df.columns

Index(['production_id', 'equipment_id', 'product_code', 'production_date',
       'start_time', 'end_time', 'target_quantity', 'actual_quantity',
       'good_quantity', 'defect_quantity', 'cycle_time', 'work_order_no',
       'lot_no', 'operator_id', 'shift', 'created_at', 'updated_at',
       'defect_rate', '목표 달성률 평균'],
      dtype='object')

In [24]:
# 여기에 코드 작성
production_df.groupby(['equipment_id','product_code']).size()

equipment_id  product_code
ASM-001       BUMPER-A         74
              DASH-C           70
              DOOR-B           90
INJ-001       BUMPER-A        104
              DASH-C           85
              DOOR-B           73
INJ-002       BUMPER-A        147
              DASH-C          122
              DOOR-B          161
PRESS-001     BUMPER-A        171
              DASH-C          139
              DOOR-B          158
PRESS-002     BUMPER-A        152
              DASH-C          167
              DOOR-B          159
dtype: int64

In [25]:
production_df.groupby(['equipment_id','product_code']).agg({'actual_quantity':'sum'})

actual_quantity
equipment_id product_code                 
ASM-001      BUMPER-A                 6804
             DASH-C                   6779
             DOOR-B                   8902
INJ-001      BUMPER-A                10858
             DASH-C                   9536
             DOOR-B                   7769
INJ-002      BUMPER-A                18053
             DASH-C                  14766
             DOOR-B                  19139
PRESS-001    BUMPER-A                18757
             DASH-C                  15670
             DOOR-B                  17642
PRESS-002    BUMPER-A                16663
             DASH-C                  17990
             DOOR-B                  17276

In [26]:
production_df.groupby(['equipment_id','product_code']).agg({'defect_rate':'mean'})

defect_rate
equipment_id product_code             
ASM-001      BUMPER-A        11.095946
             DASH-C          12.511429
             DOOR-B          12.648333
INJ-001      BUMPER-A        11.514712
             DASH-C          10.189176
             DOOR-B          10.346027
INJ-002      BUMPER-A         9.531293
             DASH-C           8.109590
             DOOR-B           8.391863
PRESS-001    BUMPER-A         9.523801
             DASH-C          10.210935
             DOOR-B          10.065759
PRESS-002    BUMPER-A        11.024539
             DASH-C          10.481257
             DOOR-B          10.546038

In [29]:
def data(x) :
    if x['equipment_id'] == 'INJ-001' :
        return True
    else :
        return False
    

In [35]:
production_df.groupby(['equipment_id','product_code']).agg({'actual_quantity':'sum'}).loc['INJ-001',]

,actual_quantity
product_code,
BUMPER-A,10858
DASH-C,9536
DOOR-B,7769


---
## 문제 5: 피벗 테이블 - 설비 x 제품 생산량 매트릭스

**시나리오**: 설비와 제품의 조합별 생산량을 한눈에 보는 표를 만드세요.

**요구사항**:
1. 피벗 테이블 생성:
   - 행: equipment_id
   - 열: product_code
   - 값: actual_quantity의 합계
   - 결측치는 0으로 채우기
2. 행/열 총계 추가 (margins=True)



In [37]:
# 여기에 코드 작성
pd.pivot_table(production_df, index='equipment_id',columns='product_code', values = 'actual_quantity',aggfunc='sum',fill_value = 0)

product_code,BUMPER-A,DASH-C,DOOR-B
equipment_id,,,
ASM-001,6804,6779,8902
INJ-001,10858,9536,7769
INJ-002,18053,14766,19139
PRESS-001,18757,15670,17642
PRESS-002,16663,17990,17276


In [38]:
# 여기에 코드 작성
pd.pivot_table(production_df, index='equipment_id',columns='product_code', values = 'actual_quantity',aggfunc='sum',fill_value = 0,margins=True)

product_code,BUMPER-A,DASH-C,DOOR-B,All
equipment_id,,,,
ASM-001,6804,6779,8902,22485
INJ-001,10858,9536,7769,28163
INJ-002,18053,14766,19139,51958
PRESS-001,18757,15670,17642,52069
PRESS-002,16663,17990,17276,51929
All,71135,64741,70728,206604


---
## 문제 6: 피벗 테이블 - 설비 x 교대조 평균 불량률

**시나리오**: 설비별로 교대조에 따라 불량률이 어떻게 다른지 파악하세요.

**요구사항**:
1. 피벗 테이블 생성:
   - 행: equipment_id
   - 열: shift
   - 값: defect_rate의 평균
2. 소수점 2자리 반올림
3. 주간과 야간의 불량률 차이가 큰 설비 찾기



In [117]:
# 여기에 코드 작성
df_shift=(pd.pivot_table(production_df, index='equipment_id',columns='shift', values = 'defect_rate',aggfunc='mean',fill_value = 0,margins=True)).round(2)# 여기에 코드 작성


In [62]:
#### 3번헷갈림

In [118]:
df_shift

shift,DAY,NIGHT,All
equipment_id,,,
ASM-001,10.67,13.56,12.12
INJ-001,9.36,12.16,10.76
INJ-002,7.20,10.20,8.70
PRESS-001,8.53,11.29,9.91
PRESS-002,9.34,12.01,10.68
All,8.82,11.63,10.22


In [121]:
df_shift.columns=['낮','밤','전체']

In [122]:
df_shift

,낮,밤,전체
equipment_id,,,
ASM-001,10.67,13.56,12.12
INJ-001,9.36,12.16,10.76
INJ-002,7.20,10.20,8.70
PRESS-001,8.53,11.29,9.91
PRESS-002,9.34,12.01,10.68
All,8.82,11.63,10.22


In [124]:
df_shift['밤']-df_shift['낮']

equipment_id
ASM-001      2.89
INJ-001      2.80
INJ-002      3.00
PRESS-001    2.76
PRESS-002    2.67
All          2.81
dtype: float64

---
## 문제 7: 불량 유형 분석 (crosstab)

**시나리오**: 제품별로 어떤 불량 유형이 많이 발생하는지 분석하세요.

**요구사항**:
1. 품질검사 데이터에서 불량(result='FAIL')만 필터링
2. 제품(product_code) x 불량코드(defect_code) 교차표 생성
3. 총계 포함
4. 각 제품의 주요 불량 유형 파악

**힌트**: 필터링 후 `pd.crosstab()`, margins=True

In [63]:
# 여기에 코드 작성
quality_df.head()

,inspection_id,production_id,equipment_id,product_code,inspection_time,inspection_type,result,defect_code,measurement_value,measurement_unit,inspector_id,lot_no,sample_size,notes,created_at
0,1,1,INJ-001,BUMPER-A,2024-01-01 09:41:18,FINAL,PASS,NaN,300.8279,mm,OP007,LOT2024010100101,1,NaN,2026-01-30 01:24:59
1,2,1,INJ-001,BUMPER-A,2024-01-01 08:17:24,FINAL,PASS,NaN,299.7696,mm,OP008,LOT2024010100101,1,NaN,2026-01-30 01:24:59
2,3,1,INJ-001,BUMPER-A,2024-01-01 08:47:26,FINAL,PASS,NaN,301.0795,mm,OP007,LOT2024010100101,1,NaN,2026-01-30 01:24:59
3,4,1,INJ-001,BUMPER-A,2024-01-01 08:33:03,FINAL,PASS,NaN,302.5384,mm,OP007,LOT2024010100101,1,NaN,2026-01-30 01:24:59
4,5,1,INJ-001,BUMPER-A,2024-01-01 09:46:23,FINAL,PASS,NaN,299.6097,mm,OP007,LOT2024010100101,1,NaN,2026-01-30 01:24:59


In [64]:
quality_df[quality_df['result']=='FAIL']

,inspection_id,production_id,equipment_id,product_code,inspection_time,inspection_type,result,defect_code,measurement_value,measurement_unit,inspector_id,lot_no,sample_size,notes,created_at
7,8,1,INJ-001,BUMPER-A,2024-01-01 08:26:47,FINAL,FAIL,D002,305.2058,mm,OP007,LOT2024010100101,1,NaN,2026-01-30 01:24:59
8,9,1,INJ-001,BUMPER-A,2024-01-01 09:30:37,FINAL,FAIL,D002,314.6991,mm,OP008,LOT2024010100101,1,NaN,2026-01-30 01:24:59
9,10,1,INJ-001,BUMPER-A,2024-01-01 08:44:05,FINAL,FAIL,D003,293.3244,mm,OP007,LOT2024010100101,1,NaN,2026-01-30 01:24:59
10,11,1,INJ-001,BUMPER-A,2024-01-01 08:35:47,FINAL,FAIL,D006,287.1234,mm,OP007,LOT2024010100101,1,NaN,2026-01-30 01:24:59
18,19,2,INJ-001,BUMPER-A,2024-01-01 21:53:40,FINAL,FAIL,D001,291.1185,mm,OP008,LOT2024010100110,1,NaN,2026-01-30 01:24:59
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37412,37413,1872,ASM-001,DOOR-B,2024-03-31 21:13:26,FINAL,FAIL,D002,608.2289,mm,OP008,LOT2024033100110,1,NaN,2026-01-30 01:25:02
37413,37414,1872,ASM-001,DOOR-B,2024-03-31 20:50:20,FINAL,FAIL,D001,609.7039,mm,OP007,LOT2024033100110,1,NaN,2026-01-30 01:25:02
37414,37415,1872,ASM-001,DOOR-B,2024-03-31 20:23:22,FINAL,FAIL,D010,590.4729,mm,OP008,LOT2024033100110,1,NaN,2026-01-30 01:25:02
37415,37416,1872,ASM-001,DOOR-B,2024-03-31 20:44:08,FINAL,FAIL,D001,609.0846,mm,OP008,LOT2024033100110,1,NaN,2026-01-30 01:25:02


In [91]:
pd.crosstab( quality_df['product_code'] , quality_df['defect_code'] ) 

defect_code,D001,D002,D003,D004,D005,D006,D007,D008,D009,D010
product_code,,,,,,,,,,
BUMPER-A,1061,1458,720,385,1063,763,390,379,732,342
DASH-C,656,963,984,339,1325,948,334,182,460,304
DOOR-B,2829,1034,748,340,734,354,205,129,358,357


In [125]:
df_crosstab=pd.crosstab( quality_df['product_code'] , quality_df['defect_code'] ,margins= True,margins_name='계') 

In [72]:
####3번 모르겠어

In [128]:
df_crosstab.index[0: -1]

Index(['BUMPER-A', 'DASH-C', 'DOOR-B'], dtype='object', name='product_code')

In [131]:
df_crosstab =df_crosstab.loc['BUMPER-A':'DOOR-B', 'D001':'D010']

In [132]:
df_crosstab

defect_code,D001,D002,D003,D004,D005,D006,D007,D008,D009,D010
product_code,,,,,,,,,,
BUMPER-A,1061,1458,720,385,1063,763,390,379,732,342
DASH-C,656,963,984,339,1325,948,334,182,460,304
DOOR-B,2829,1034,748,340,734,354,205,129,358,357


In [136]:
 for name in df_crosstab.index[:] :
     index = df_crosstab.loc[name, 'D001':'D010'].argmax() #인덱스 반
     print (f'{name}의 불량이 가장 높은 유형은? : {df_crosstab.columns[0:-1][index]}')

BUMPER-A의 불량이 가장 높은 유형은? : D002
DASH-C의 불량이 가장 높은 유형은? : D005
DOOR-B의 불량이 가장 높은 유형은? : D001


In [137]:
df_cosstab[index]

NameError: name 'df_cosstab' is not defined

---
## 문제 8: 월별 생산 추이 분석

**시나리오**: 월별 생산량과 품질 추이를 분석하여 트렌드를 파악하세요.

**요구사항**:
1. production_date에서 년-월 추출 (dt.to_period('M'))
2. 월별로 다음 집계:
   - 생산 건수
   - 총 생산량
   - 평균 생산량
   - 평균 불량률
3. 시간 순서로 정렬
4. 처음 12개월 데이터 출력



In [74]:
# 여기에 코드 작성
production_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1872 entries, 0 to 1871
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   production_id    1872 non-null   int64         
 1   equipment_id     1872 non-null   object        
 2   product_code     1872 non-null   object        
 3   production_date  1872 non-null   datetime64[ns]
 4   start_time       1872 non-null   object        
 5   end_time         1872 non-null   object        
 6   target_quantity  1872 non-null   int64         
 7   actual_quantity  1872 non-null   int64         
 8   good_quantity    1872 non-null   int64         
 9   defect_quantity  1872 non-null   int64         
 10  cycle_time       1872 non-null   float64       
 11  work_order_no    1872 non-null   object        
 12  lot_no           1872 non-null   object        
 13  operator_id      1872 non-null   object        
 14  shift            1872 non-null   object 

In [75]:
production_df.head()

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,cycle_time,work_order_no,lot_no,operator_id,shift,created_at,updated_at,defect_rate,목표 달성률 평균
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,73.73,WO202401019935,LOT2024010100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,4.94,83.505155
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,70.56,WO202401012535,LOT2024010100110,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,7.69,93.975904
2,3,INJ-002,BUMPER-A,2024-01-01,2024-01-01 10:12:00,2024-01-01 13:16:28,149,135,132,3,81.99,WO202401018359,LOT2024010100201,OP001,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,2.22,90.604027
3,4,INJ-002,DASH-C,2024-01-01,2024-01-01 12:48:00,2024-01-01 15:16:31,100,92,90,2,96.87,WO202401016574,LOT2024010100202,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,2.17,92.000000
4,5,INJ-002,DOOR-B,2024-01-01,2024-01-01 20:48:00,2024-01-01 23:12:13,123,129,122,7,67.08,WO202401012674,LOT2024010100210,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,5.43,104.878049


In [79]:
production_df.tail()

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,cycle_time,work_order_no,lot_no,operator_id,shift,created_at,updated_at,defect_rate,목표 달성률 평균,생산 월
1867,1868,PRESS-002,BUMPER-A,2024-03-31,2024-03-31 20:19:00,2024-03-31 23:25:19,150,144,119,25,77.63,WO202403317101,LOT2024033100210,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,17.36,96.000000,2024-03
1868,1869,PRESS-002,DASH-C,2024-03-31,2024-04-01 00:15:00,2024-04-01 02:59:58,136,130,109,21,76.15,WO202403318434,LOT2024033100211,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,16.15,95.588235,2024-03
1869,1870,PRESS-002,BUMPER-A,2024-03-31,2024-04-01 05:53:00,2024-04-01 07:26:15,84,80,66,14,69.95,WO202403317294,LOT2024033100212,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,17.50,95.238095,2024-03
1870,1871,ASM-001,BUMPER-A,2024-03-31,2024-03-31 10:24:00,2024-03-31 13:25:41,143,121,101,20,90.10,WO202403317268,LOT2024033100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,16.53,84.615385,2024-03
1871,1872,ASM-001,DOOR-B,2024-03-31,2024-03-31 20:14:00,2024-03-31 22:16:42,106,90,74,16,81.80,WO202403318062,LOT2024033100110,OP005,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,17.78,84.905660,2024-03


In [77]:
production_df['생산 월']=production_df['production_date'].dt.to_period('M')

In [78]:
production_df.groupby('생산 월').agg({'equipment_id':'count','actual_quantity':'sum','actual_quantity':'mean','defect_rate':'mean'})

,equipment_id,actual_quantity,defect_rate
생산 월,,,
2024-01,626,111.579872,5.304601
2024-02,602,110.282392,10.364003
2024-03,644,109.262422,14.871227


---
## 문제 9: Transform을 이용한 설비별 표준화

**시나리오**: 각 설비의 생산량을 설비별 평균과 비교하여 성과를 평가하세요.

**요구사항**:
1. 설비별 평균 생산량을 계산하여 새 컬럼으로 추가 (transform)
2. 설비별 표준편차를 계산하여 새 컬럼으로 추가
3. 각 생산 건의 표준화 점수(Z-score) 계산:
   - Z-score = (실제값 - 평균) / 표준편차
4. Z-score가 2 이상인 생산 건 찾기 (매우 높은 생산량)



In [80]:
# 여기에 코드 작성
production_df.head()

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,cycle_time,work_order_no,lot_no,operator_id,shift,created_at,updated_at,defect_rate,목표 달성률 평균,생산 월
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,73.73,WO202401019935,LOT2024010100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,4.94,83.505155,2024-01
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,70.56,WO202401012535,LOT2024010100110,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,7.69,93.975904,2024-01
2,3,INJ-002,BUMPER-A,2024-01-01,2024-01-01 10:12:00,2024-01-01 13:16:28,149,135,132,3,81.99,WO202401018359,LOT2024010100201,OP001,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,2.22,90.604027,2024-01
3,4,INJ-002,DASH-C,2024-01-01,2024-01-01 12:48:00,2024-01-01 15:16:31,100,92,90,2,96.87,WO202401016574,LOT2024010100202,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,2.17,92.000000,2024-01
4,5,INJ-002,DOOR-B,2024-01-01,2024-01-01 20:48:00,2024-01-01 23:12:13,123,129,122,7,67.08,WO202401012674,LOT2024010100210,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,5.43,104.878049,2024-01


In [83]:
production_df['설비별 평균 생산량']=production_df.groupby('equipment_id')['actual_quantity'].transform('mean')

In [84]:
production_df['설비별 평균 표준편차']=production_df.groupby('equipment_id')['actual_quantity'].transform('std')

In [85]:
production_df.head()

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,...,lot_no,operator_id,shift,created_at,updated_at,defect_rate,목표 달성률 평균,생산 월,설비별 평균 생산량,설비별 평균 표준편차
0,1,INJ-001,BUMPER-A,2024-01-01,2024-01-01 08:14:00,2024-01-01 09:53:32,97,81,77,4,...,LOT2024010100101,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,4.94,83.505155,2024-01,107.492366,20.022352
1,2,INJ-001,BUMPER-A,2024-01-01,2024-01-01 21:02:00,2024-01-01 22:33:43,83,78,72,6,...,LOT2024010100110,OP006,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,7.69,93.975904,2024-01,107.492366,20.022352
2,3,INJ-002,BUMPER-A,2024-01-01,2024-01-01 10:12:00,2024-01-01 13:16:28,149,135,132,3,...,LOT2024010100201,OP001,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,2.22,90.604027,2024-01,120.832558,22.112656
3,4,INJ-002,DASH-C,2024-01-01,2024-01-01 12:48:00,2024-01-01 15:16:31,100,92,90,2,...,LOT2024010100202,OP003,DAY,2026-01-30 00:42:48,2026-01-30 00:42:48,2.17,92.000000,2024-01,120.832558,22.112656
4,5,INJ-002,DOOR-B,2024-01-01,2024-01-01 20:48:00,2024-01-01 23:12:13,123,129,122,7,...,LOT2024010100210,OP004,NIGHT,2026-01-30 00:42:48,2026-01-30 00:42:48,5.43,104.878049,2024-01,120.832558,22.112656


In [86]:
production_df['z_score']=(production_df['actual_quantity']-production_df['설비별 평균 생산량'])/production_df['설비별 평균 표준편차']

In [90]:
production_df[production_df['z_score']>=2]

,production_id,equipment_id,product_code,production_date,start_time,end_time,target_quantity,actual_quantity,good_quantity,defect_quantity,...,operator_id,shift,created_at,updated_at,defect_rate,목표 달성률 평균,생산 월,설비별 평균 생산량,설비별 평균 표준편차,z_score


---
## 문제 10: 종합 설비 성능 대시보드

**시나리오**: 설비별 성능을 종합적으로 평가하는 대시보드 데이터를 만드세요.

**요구사항**:
1. 설비별로 다음 지표 모두 계산:
   - 생산 건수
   - 총 생산량
   - 평균 생산량
   - 생산량 표준편차
   - 평균 불량률
   - 평균 사이클 타임
   - 평균 목표 달성률
2. 성능 점수 계산 (사용자 정의):
   - 성능점수 = (평균생산량 / 평균사이클타임) * (100 - 평균불량률)
3. 성능 점수 기준 순위 매기기
4. 성능 점수 상위 5개 설비 출력



In [96]:
production_df.columns

Index(['production_id', 'equipment_id', 'product_code', 'production_date',
       'start_time', 'end_time', 'target_quantity', 'actual_quantity',
       'good_quantity', 'defect_quantity', 'cycle_time', 'work_order_no',
       'lot_no', 'operator_id', 'shift', 'created_at', 'updated_at',
       'defect_rate', '목표 달성률 평균', '생산 월', '설비별 평균 생산량', '설비별 평균 표준편차',
       'z_score'],
      dtype='object')

In [99]:
production_df['목표 달성률']=production_df['actual_quantity']/production_df['target_quantity'] *100

In [ ]:
production_df['성능점수']=production_df['actual_quantity']/production_df['target_quantity'] *100

In [102]:
# 여기에 코드 작성
df_edited=production_df.groupby('equipment_id').agg({'production_id':'count',
                                          'actual_quantity':['sum','mean','std'],
                                           'defect_rate' : 'mean',
                                           'cycle_time' : 'mean',
                                           '목표 달성률': 'mean'
                                                                                      
                                          })

In [104]:
df_edited

production_id actual_quantity                        defect_rate  \
                     count             sum        mean        std        mean   
equipment_id                                                                    
ASM-001                234           22485   96.089744  16.575114   12.116453   
INJ-001                262           28163  107.492366  20.022352   10.759046   
INJ-002                430           51958  120.832558  22.112656    8.701302   
PRESS-001              468           52069  111.258547  20.685662    9.910855   
PRESS-002              478           51929  108.638075  20.195916   10.675565   

             cycle_time      목표 달성률  
                   mean        mean  
equipment_id                         
ASM-001       94.427137   83.716011  
INJ-001       71.714695   93.584523  
INJ-002       77.825558  103.688416  
PRESS-001     73.568333   96.670964  
PRESS-002     72.555900   94.705213

In [105]:
df_edited.columns = [
    'production_count',
    'total_quantity',
    'avg_quantity',
    'std_quantity',
    'avg_defect_rate',
    'avg_cycle_time',
    'avg_target_achievement'
]


In [106]:
df_edited

,production_count,total_quantity,avg_quantity,std_quantity,avg_defect_rate,avg_cycle_time,avg_target_achievement
equipment_id,,,,,,,
ASM-001,234,22485,96.089744,16.575114,12.116453,94.427137,83.716011
INJ-001,262,28163,107.492366,20.022352,10.759046,71.714695,93.584523
INJ-002,430,51958,120.832558,22.112656,8.701302,77.825558,103.688416
PRESS-001,468,52069,111.258547,20.685662,9.910855,73.568333,96.670964
PRESS-002,478,51929,108.638075,20.195916,10.675565,72.555900,94.705213


In [107]:
df_edited['score'] = (
    (df_edited['avg_quantity'] / df_edited['avg_cycle_time']) *
    (100 - df_edited['avg_defect_rate'])
)

In [114]:
df_edited.sort_values(by='score', ascending=False)

,production_count,total_quantity,avg_quantity,std_quantity,avg_defect_rate,avg_cycle_time,avg_target_achievement,score
equipment_id,,,,,,,,
INJ-002,430,51958,120.832558,22.112656,8.701302,77.825558,103.688416,141.751058
PRESS-001,468,52069,111.258547,20.685662,9.910855,73.568333,96.670964,136.243231
INJ-001,262,28163,107.492366,20.022352,10.759046,71.714695,93.584523,133.762284
PRESS-002,478,51929,108.638075,20.195916,10.675565,72.555900,94.705213,133.745633
ASM-001,234,22485,96.089744,16.575114,12.116453,94.427137,83.716011,89.430939


In [116]:
df_edited['score'].sort_values()

equipment_id
ASM-001       89.430939
PRESS-002    133.745633
INJ-001      133.762284
PRESS-001    136.243231
INJ-002      141.751058
Name: score, dtype: float64

---
## 수고하셨습니다!

### 학습 체크리스트
- [ ] groupby로 단일/다중 그룹화
- [ ] agg로 여러 집계 함수 동시 적용
- [ ] 명확한 컬럼명으로 집계 결과 생성
- [ ] pivot_table로 행/열 구조 변환
- [ ] crosstab으로 교차표 생성
- [ ] value_counts로 빈도 계산
- [ ] transform으로 그룹별 계산 결과 추가
- [ ] 복합 지표 계산 및 성능 평가

